In [3]:
import pandas as pd
import numpy as np
import random
import os

from sklearn.model_selection import train_test_split
import lightgbm as lgb
import optuna

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModel
import faiss
from tqdm import tqdm
import json
from pathlib import Path

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [4]:
transactions = pd.read_csv('drive/MyDrive/transactions_features.csv', parse_dates=['t_dat'])
articles = pd.read_csv('drive/MyDrive/articles_features.csv')
customers = pd.read_csv('drive/MyDrive/customers_features.csv')

max_date = transactions['t_dat'].max()
test_start = max_date - pd.Timedelta(days=7)
val_start = test_start - pd.Timedelta(days=7)

train_data = transactions[transactions['t_dat'] < val_start].copy()
val_data = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)].copy()
test_data = transactions[transactions['t_dat'] >= test_start].copy()

In [5]:
text_columns = [
    'prod_name', 'product_type_name', 'product_group_name',
    'graphical_appearance_name', 'colour_group_name',
    'department_name', 'index_name', 'detail_desc'
]

for col in text_columns:
    articles[col] = articles[col].fillna('')

articles['text_description'] = (
    articles['prod_name'].astype(str) + ". " +
    "Category: " + articles['product_group_name'].astype(str) + " - " + articles['product_type_name'].astype(str) + ". " +
    "Style: " + articles['graphical_appearance_name'].astype(str) + ", Color: " + articles['colour_group_name'].astype(str) + ". " +
    "Department: " + articles['department_name'].astype(str) + " (" + articles['index_name'].astype(str) + "). " +
    "Description: " + articles['detail_desc'].astype(str)
)

print(articles['text_description'].iloc[0])

Strap top. Category: Garment Upper body - Vest top. Style: Solid, Color: White. Department: Jersey Basic (Ladieswear). Description: Jersey top with narrow shoulder straps.


In [6]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(83828, 312, padding_idx=0)
    (position_embeddings): Embedding(2048, 312)
    (token_type_embeddings): Embedding(2, 312)
    (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-2): 3 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=312, out_features=312, bias=True)
            (key): Linear(in_features=312, out_features=312, bias=True)
            (value): Linear(in_features=312, out_features=312, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=312, out_features=312, bias=True)
            (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
   

In [7]:
def extract_bert_embeddings(text_list, batch_size=256):
    all_embeddings = []
    for i in tqdm(range(0, len(text_list), batch_size), desc="BERT Inference"):
        batch_texts = text_list[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        all_embeddings.append(batch_embeddings)

    return np.vstack(all_embeddings)

article_texts = articles['text_description'].tolist()
item_embeddings = extract_bert_embeddings(article_texts, batch_size=512)
print(item_embeddings.shape)

BERT Inference: 100%|██████████| 58/58 [00:26<00:00,  2.15it/s]

(29237, 312)


In [8]:
dimension = item_embeddings.shape[1]

item_embeddings_norm = item_embeddings.copy()
faiss.normalize_L2(item_embeddings_norm)

index = faiss.IndexFlatIP(dimension)
index.add(item_embeddings_norm)

item_id_to_idx = {idx: i for i, idx in enumerate(articles['article_id'].values)}
idx_to_item_id = {i: idx for i, idx in enumerate(articles['article_id'].values)}

train_data['item_idx'] = train_data['article_id'].map(item_id_to_idx)
train_data = train_data.dropna(subset=['item_idx'])
train_data['item_idx'] = train_data['item_idx'].astype(int)

user_history = train_data.groupby('customer_id')['item_idx'].apply(list).to_dict()
val_actual_purchases = val_data.groupby('customer_id')['article_id'].apply(list).to_dict()
target_users = list(val_actual_purchases.keys())

In [9]:
query_embs = []
valid_users = []
for user in target_users:
    if user in user_history:
        user_item_indices = user_history[user]
        user_profile_vector = item_embeddings[user_item_indices].mean(axis=0)
        query_embs.append(user_profile_vector)
        valid_users.append(user)

query_embs = np.array(query_embs).astype('float32')
faiss.normalize_L2(query_embs)

In [10]:
Distances, Indices = index.search(query_embs, 100)

candidate_rows = []
for i, user in enumerate(tqdm(valid_users, desc="Сборка датасета")):
    for rank, (idx, score) in enumerate(zip(Indices[i], Distances[i])):
        if idx in idx_to_item_id:
            candidate_rows.append({
                'customer_id': user,
                'article_id': idx_to_item_id[idx],
                'faiss_score': score,
                'faiss_rank': rank + 1
            })

df_candidates = pd.DataFrame(candidate_rows)

Сборка датасета: 100%|██████████| 24690/24690 [00:02<00:00, 9431.69it/s]


In [11]:
val_actual_purchases = val_data[['customer_id', 'article_id']].drop_duplicates()
val_actual_purchases['target'] = 1

df_ranker = df_candidates.merge(val_actual_purchases, on=['customer_id', 'article_id'], how='left')
df_ranker['target'] = df_ranker['target'].fillna(0).astype(int)

In [12]:
item_features = train_data.groupby('article_id').agg(
    avg_price=('price', 'mean'),
    popularity=('customer_id', 'count')
).reset_index()

item_features['avg_price'] = item_features['avg_price'].fillna(item_features['avg_price'].median())

df_ranker = df_ranker.merge(item_features, on='article_id', how='left')
df_ranker['avg_price'] = df_ranker['avg_price'].fillna(df_ranker['avg_price'].mean())
df_ranker['popularity'] = df_ranker['popularity'].fillna(0)


In [13]:
if not customers.empty:
    df_ranker = df_ranker.merge(customers[['customer_id', 'age']], on='customer_id', how='left')
    df_ranker['age'] = df_ranker['age'].fillna(df_ranker['age'].median())

user_features = train_data.groupby('customer_id').agg(
    user_avg_spend=('price', 'mean')
).reset_index()

df_ranker = df_ranker.merge(user_features, on='customer_id', how='left')

global_mean_spend = train_data['price'].mean()
df_ranker['user_avg_spend'] = df_ranker['user_avg_spend'].fillna(global_mean_spend)

df_ranker['price_diff'] = df_ranker['avg_price'] - df_ranker['user_avg_spend']

df_ranker.head()

,customer_id,article_id,faiss_score,faiss_rank,target,avg_price,popularity,age,user_avg_spend,price_diff
0,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,874704002,0.979446,1,0,0.049921,270.0,44.0,0.02802,0.021901
1,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,826492007,0.978105,2,0,0.020643,58.0,44.0,0.02802,-0.007377
2,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,537119003,0.977484,3,0,0.059305,3.0,44.0,0.02802,0.031285
3,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,826492001,0.977023,4,0,0.032345,80.0,44.0,0.02802,0.004325
4,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,896851001,0.976808,5,0,0.033312,438.0,44.0,0.02802,0.005292


In [14]:
features = [
    'faiss_score',
    'faiss_rank',
    'avg_price',
    'popularity',
    'age',
    'user_avg_spend',
    'price_diff'
]

X = df_ranker[features]
y = df_ranker['target']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

In [15]:
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 5,
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1
}

train_dataset = lgb.Dataset(X_train, label=y_train)
valid_dataset = lgb.Dataset(X_valid, label=y_valid, reference=train_dataset)

In [16]:
ranker_model = lgb.train(
    lgb_params,
    train_dataset,
    num_boost_round=300,
    valid_sets=[valid_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(50)]
)

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.877315


In [17]:
importances = pd.DataFrame({
    'feature': features,
    'importance': ranker_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

importances

,feature,importance
3,popularity,1.422574e+08
1,faiss_rank,5.440948e+07
0,faiss_score,1.060931e+07
2,avg_price,4.485855e+06
6,price_diff,2.171273e+06
4,age,0.000000e+00
5,user_avg_spend,0.000000e+00


In [18]:
features = [
    'faiss_score',
    'faiss_rank',
    'avg_price',
    'popularity',
    'price_diff'
]

X = df_ranker[features]
y = df_ranker['target']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

train_dataset = lgb.Dataset(X_train, label=y_train)
valid_dataset = lgb.Dataset(X_valid, label=y_valid, reference=train_dataset)

ranker_model = lgb.train(
    lgb_params,
    train_dataset,
    num_boost_round=300,
    valid_sets=[valid_dataset],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(50)]
)

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.877315


In [19]:
df_ranker['predict_prob'] = ranker_model.predict(df_ranker[features])

df_ranker_sorted = df_ranker.sort_values(['customer_id', 'predict_prob'], ascending=[True, False])


top_20_recommendations = df_ranker_sorted.groupby('customer_id').head(20)

preds_dict_boosting = top_20_recommendations.groupby('customer_id')['article_id'].apply(list).to_dict()

df_ranker_sorted[['customer_id', 'article_id', 'faiss_rank', 'price_diff', 'predict_prob', 'target']].head(10)

,customer_id,article_id,faiss_rank,price_diff,predict_prob,target
0,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,874704002,1,0.021901,1.000000,0
4,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,896851001,5,0.005292,1.000000,0
6,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,873276001,7,0.005020,1.000000,0
14,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,873276004,15,0.005242,0.999998,0
15,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,873276003,16,0.005057,0.999998,0
1,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,826492007,2,-0.007377,0.999997,0
3,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,826492001,4,0.004325,0.999997,0
10,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,826492003,11,0.004096,0.999988,0
48,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,874754004,49,0.005159,0.999972,0
88,0001d44dbe7f6c4b35200abdb052c77a87596fe1bdcc37...,928040002,89,0.038168,0.999972,0


In [20]:
def calculate_metrics(actual_dict, predicted_dict, k=20):
    precisions, recalls = [], []
    for user, actual_items in actual_dict.items():
        if user not in predicted_dict:
            continue
        actual_set = set(actual_items)
        predicted_set = set(predicted_dict[user][:k])
        hits = len(actual_set & predicted_set)
        precisions.append(hits / k)
        recalls.append(hits / len(actual_set) if len(actual_set) > 0 else 0)

    if not precisions:
        return 0.0, 0.0
    return np.mean(precisions), np.mean(recalls)

In [21]:
actual_dict = val_data.groupby('customer_id')['article_id'].apply(list).to_dict()

p_boost, r_boost = calculate_metrics(actual_dict, preds_dict_boosting, k=20)

print(p_boost)
print(r_boost)

0.008699878493317131
0.08393085129877073


In [22]:
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'is_unbalance': True,
        'random_state': 42,
        'verbose': -1,
        'feature_pre_filter': False,

        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100)
    }

    train_data_optuna = lgb.Dataset(
        X_train,
        label=y_train,
        params={'feature_pre_filter': False},
        free_raw_data=False
    )
    valid_data_optuna = lgb.Dataset(
        X_valid,
        label=y_valid,
        reference=train_data_optuna,
        free_raw_data=False
    )

    model = lgb.train(
        params,
        train_data_optuna,
        num_boost_round=200,
        valid_sets=[valid_data_optuna],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
    )

    return model.best_score['valid_0']['auc']

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)


best_params = study.best_params
for key, value in best_params.items():
    print(f"  {key}: {value}")

print(study.best_value)

  0%|          | 0/20 [00:00<?, ?it/s]

  learning_rate: 0.010236397318562285
  num_leaves: 17
  max_depth: 8
  min_child_samples: 67
0.8903104512501314


In [23]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1,
    'feature_pre_filter': False,
    'learning_rate': 0.010227680947327136,
    'num_leaves': 21,
    'max_depth': 5,
    'min_child_samples': 69
}

features = ['faiss_score', 'faiss_rank', 'avg_price', 'popularity', 'price_diff']

train_data_final = lgb.Dataset(X_train, label=y_train, params={'feature_pre_filter': False}, free_raw_data=False)
valid_data_final = lgb.Dataset(X_valid, label=y_valid, reference=train_data_final, free_raw_data=False)

final_model = lgb.train(
    params,
    train_data_final,
    num_boost_round=500,
    valid_sets=[valid_data_final],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

df_ranker['predict_prob'] = final_model.predict(df_ranker[features])
df_ranker_sorted = df_ranker.sort_values(['customer_id', 'predict_prob'], ascending=[True, False])

top_20_final = df_ranker_sorted.groupby('customer_id').head(20)
preds_dict_final = top_20_final.groupby('customer_id')['article_id'].apply(list).to_dict()

actual_dict = val_data.groupby('customer_id')['article_id'].apply(list).to_dict()
p_val, r_val = calculate_metrics(actual_dict, preds_dict_final, k=20)

print(p_val)
print(r_val)

0.009066423653300933
0.08678555253738512


In [24]:
test_users = test_data['customer_id'].unique()

history_before_test = transactions[transactions['t_dat'] < test_data['t_dat'].min()].copy()
history_before_test['item_idx'] = history_before_test['article_id'].map(item_id_to_idx)
history_before_test = history_before_test.dropna(subset=['item_idx'])
history_before_test['item_idx'] = history_before_test['item_idx'].astype(int)

user_history_dict = history_before_test.groupby('customer_id')['item_idx'].apply(list).to_dict()

active_known_users = [user for user in test_users if user in user_history_dict]

query_embs_test = []
for user in active_known_users:
    user_vectors = item_embeddings[user_history_dict[user]]
    query_embs_test.append(user_vectors.mean(axis=0))

query_embs_test = np.array(query_embs_test).astype('float32')
faiss.normalize_L2(query_embs_test)

In [25]:
Distances, Indices = index.search(query_embs_test, 100)

candidate_rows = []
for i, user in enumerate(active_known_users):
    for rank, (idx, score) in enumerate(zip(Indices[i], Distances[i])):
        if idx in idx_to_item_id:
            candidate_rows.append({
                'customer_id': user,
                'article_id': idx_to_item_id[idx],
                'faiss_score': score,
                'faiss_rank': rank + 1
            })

df_test_ranker = pd.DataFrame(candidate_rows)

user_features_test = history_before_test.groupby('customer_id').agg(
    user_avg_spend=('price', 'mean')
).reset_index()

df_test_ranker = df_test_ranker.merge(item_features, on='article_id', how='left')
df_test_ranker = df_test_ranker.merge(user_features_test, on='customer_id', how='left')

df_test_ranker['avg_price'] = df_test_ranker['avg_price'].fillna(df_test_ranker['avg_price'].median())
df_test_ranker['popularity'] = df_test_ranker['popularity'].fillna(0)
df_test_ranker['user_avg_spend'] = df_test_ranker['user_avg_spend'].fillna(df_test_ranker['avg_price'].median())
df_test_ranker['price_diff'] = df_test_ranker['avg_price'] - df_test_ranker['user_avg_spend']

features = ['faiss_score', 'faiss_rank', 'avg_price', 'popularity', 'price_diff']
df_test_ranker['predict_prob'] = final_model.predict(df_test_ranker[features])

df_test_ranker_sorted = df_test_ranker.sort_values(['customer_id', 'predict_prob'], ascending=[True, False])
top_20_test = df_test_ranker_sorted.groupby('customer_id').head(20)
preds_dict_test = top_20_test.groupby('customer_id')['article_id'].apply(list).to_dict()

actual_dict_test = test_data.groupby('customer_id')['article_id'].apply(list).to_dict()

p_test, r_test = calculate_metrics(actual_dict_test, preds_dict_test, k=20)

print(p_test)
print(r_test)

0.006909291246718735
0.06636238307699711


In [29]:
all_metrics = {
    "BERT_LightGBM": {
        "val_precision_20": float(p_val),
        "val_recall_20": float(r_val),
        "test_precision_20": float(p_test),
        "test_recall_20": float(r_test),
    }
}
with open(ARTIFACTS_DIR/'bert_lgbm_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=4)

In [28]:
bert_lgbm_dir = os.path.join(ARTIFACTS_DIR, "bert_lgbm")
os.makedirs(bert_lgbm_dir, exist_ok=True)

lgb_path = os.path.join(bert_lgbm_dir, 'lgbm_model.txt')
final_model.save_model(lgb_path)

faiss_path = os.path.join(bert_lgbm_dir, 'faiss_index.bin')
faiss.write_index(index, faiss_path)

emb_path = os.path.join(bert_lgbm_dir, 'item_embeddings.npy')
np.save(emb_path, item_embeddings)


mapping_dicts = {
    'item_id_to_idx': {str(k): int(v) for k, v in item_id_to_idx.items()},
    'idx_to_item_id': {str(k): str(v) for k, v in idx_to_item_id.items()}
}
map_path = os.path.join(bert_lgbm_dir, 'mapping_dicts.json')
with open(map_path, 'w', encoding='utf-8') as f:
    json.dump(mapping_dicts, f)

features_path = os.path.join(bert_lgbm_dir, 'item_features.csv')
item_features.to_csv(features_path, index=False)

history_json = {str(k): [int(x) for x in v] for k, v in user_history_dict.items()}
history_path = os.path.join(bert_lgbm_dir, 'user_history_dict.json')
with open(history_path, 'w', encoding='utf-8') as f:
    json.dump(history_json, f)